# Notebook 06: Ablation Studies

## Overview
This notebook quantifies the contribution of each key design choice in the Hybrid-Lift framework by systematically removing one component at a time and measuring the impact on out-of-sample performance.

## Ablation Tests
1. **First Differences vs Levels**: Train the champion architecture on absolute levels instead of first differences.
2. **With MBC vs Without MBC**: Reconstruct validation levels without the Mean-Bias Correction.
3. **Multi-Population vs Single-Population**: Train a model on only the common factor $K_t$ (1D) instead of all 7 factors jointly.


## 6.1: Environment Setup and Data Loading

In [1]:
# Reproducibility: Set global seed before any other imports
import sys
sys.path.append('../src')
from reproducibility import set_global_seed
set_global_seed()

import os
import logging
import numpy as np
import pandas as pd
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import tensorflow as tf
tf.get_logger().setLevel(logging.ERROR)

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import joblib

# Load data
DATA_PATH = '../data/processed/benchmarking_factors.npz'
with np.load(DATA_PATH, allow_pickle=True) as data:
    kt_common = data['kt_common']
    kt_specific = data['kt_specific_matrix']
    years = data['years']
    country_codes = data['countries']

full_features = np.column_stack([kt_common, kt_specific])
n_features = full_features.shape[1]
LOOKBACK = 10

# Load champion model for reference
champion_model = tf.keras.models.load_model('../models/mortality_lstm_champion.keras', compile=False)
champion_scaler = joblib.load('../models/data_scaler.pkl')

# Load validation results for MBC ablation
with np.load('../data/processed/validation_results.npz') as data:
    y_true_val = data['y_true']
    y_pred_val = data['y_pred']
    val_years = data['years']

print(f'Data loaded: {len(years)} years, {n_features} features')
print(f'Validation period: {val_years[0]}-{val_years[-1]}')

/Users/darindor2101/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[Reproducibility] Global seed set to 42
Data loaded: 65 years, 7 features
Validation period: 2011-2020


## 6.2: Ablation 1 — First Differences vs Absolute Levels

In [2]:
# Train a model on LEVELS (not differences)
# Use the same architecture as champion: 32/16 units, dropout 0.2, lr 0.001

from reproducibility import set_global_seed
set_global_seed()  # Reset seed for fair comparison

# Prepare LEVEL-based sequences
scaler_levels = StandardScaler()
train_split_idx = int(len(full_features) * 0.85)
scaler_levels.fit(full_features[:train_split_idx])
levels_scaled = scaler_levels.transform(full_features)

def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback])
    return np.array(X), np.array(y)

X_levels, y_levels = create_sequences(levels_scaled, LOOKBACK)
X_train_lev = X_levels[:train_split_idx - LOOKBACK]
y_train_lev = y_levels[:train_split_idx - LOOKBACK]
X_val_lev = X_levels[train_split_idx - LOOKBACK:]
y_val_lev = y_levels[train_split_idx - LOOKBACK:]

# Build same architecture
def build_champion_arch(n_features):
    inputs = tf.keras.Input(shape=(LOOKBACK, n_features))
    x = tf.keras.layers.LSTM(32, return_sequences=True)(inputs)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.LSTM(16, return_sequences=False)(x)
    outputs = tf.keras.layers.Dense(n_features, activation='linear')(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

model_levels = build_champion_arch(n_features)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

print('Training model on LEVELS...')
model_levels.fit(X_train_lev, y_train_lev, epochs=150, batch_size=8,
                 validation_data=(X_val_lev, y_val_lev),
                 callbacks=[early_stop], verbose=0)

# Evaluate: predict on validation set
y_pred_levels = model_levels.predict(X_val_lev, verbose=0)

# RMSE in original space
y_true_orig_lev = scaler_levels.inverse_transform(y_val_lev)
y_pred_orig_lev = scaler_levels.inverse_transform(y_pred_levels)
rmse_levels = np.sqrt(mean_squared_error(y_true_orig_lev[:, 0], y_pred_orig_lev[:, 0]))

# Compare with champion (differences) - RMSE on Kt common factor
# Champion predicts differences, so we need level-reconstructed RMSE
# Use the validation results already computed
bias = np.mean(y_true_val - y_pred_val, axis=0)
last_train_level = full_features[train_split_idx - 1]

# Reconstruct levels from champion differences WITH MBC
curr = last_train_level.copy()
pred_levels_champ = []
for t in range(len(val_years)):
    curr = curr + champion_scaler.inverse_transform(y_pred_val[t:t+1])[0] + bias
    pred_levels_champ.append(curr.copy())
pred_levels_champ = np.array(pred_levels_champ)

true_levels_val = full_features[train_split_idx:train_split_idx + len(val_years)]
rmse_diffs_champion = np.sqrt(mean_squared_error(true_levels_val[:, 0], pred_levels_champ[:, 0]))

print(f'\n--- ABLATION 1: First Differences vs Levels ---')
print(f'Champion (Differences + MBC) RMSE on Kt: {rmse_diffs_champion:.4f}')
print(f'Ablation (Levels) RMSE on Kt: {rmse_levels:.4f}')
improvement = (rmse_levels - rmse_diffs_champion) / rmse_levels * 100
print(f'Improvement from using differences: +{improvement:.1f}%')

[Reproducibility] Global seed set to 42
Training model on LEVELS...

--- ABLATION 1: First Differences vs Levels ---
Champion (Differences + MBC) RMSE on Kt: 15.8981
Ablation (Levels) RMSE on Kt: 30.9305
Improvement from using differences: +48.6%


## 6.3: Ablation 2 — With MBC vs Without MBC

In [3]:
# Reconstruct levels WITHOUT MBC (bias = 0)
curr_no_mbc = last_train_level.copy()
pred_levels_no_mbc = []
for t in range(len(val_years)):
    curr_no_mbc = curr_no_mbc + champion_scaler.inverse_transform(y_pred_val[t:t+1])[0]
    pred_levels_no_mbc.append(curr_no_mbc.copy())
pred_levels_no_mbc = np.array(pred_levels_no_mbc)

rmse_no_mbc = np.sqrt(mean_squared_error(true_levels_val[:, 0], pred_levels_no_mbc[:, 0]))

print(f'--- ABLATION 2: With MBC vs Without MBC ---')
print(f'With MBC RMSE on Kt: {rmse_diffs_champion:.4f}')
print(f'Without MBC RMSE on Kt: {rmse_no_mbc:.4f}')
improvement_mbc = (rmse_no_mbc - rmse_diffs_champion) / rmse_no_mbc * 100
print(f'Improvement from MBC: +{improvement_mbc:.1f}%')

--- ABLATION 2: With MBC vs Without MBC ---
With MBC RMSE on Kt: 15.8981
Without MBC RMSE on Kt: 19.5271
Improvement from MBC: +18.6%


## 6.4: Ablation 3 — Multi-Population (7D) vs Single-Population (1D)

In [4]:
# Train a model on ONLY the common factor Kt (1 feature)
from reproducibility import set_global_seed
set_global_seed()  # Reset seed for fair comparison

# Prepare single-factor differences
single_features = kt_common.reshape(-1, 1)
diff_single = np.diff(single_features, axis=0)

scaler_single = StandardScaler()
scaler_single.fit(diff_single[:train_split_idx])
diff_single_scaled = scaler_single.transform(diff_single)

X_single, y_single = create_sequences(diff_single_scaled, LOOKBACK)
X_train_s = X_single[:train_split_idx - LOOKBACK]
y_train_s = y_single[:train_split_idx - LOOKBACK]
X_val_s = X_single[train_split_idx - LOOKBACK:]
y_val_s = y_single[train_split_idx - LOOKBACK:]

# Build single-output model (same depth, 1 feature)
model_single = build_champion_arch(1)

print('Training single-population model (Kt only)...')
model_single.fit(X_train_s, y_train_s, epochs=150, batch_size=8,
                 validation_data=(X_val_s, y_val_s),
                 callbacks=[tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)],
                 verbose=0)

# Predict and reconstruct levels
y_pred_single = model_single.predict(X_val_s, verbose=0)
y_true_single = y_val_s

# MBC for single model
bias_single = np.mean(scaler_single.inverse_transform(y_true_single) - scaler_single.inverse_transform(y_pred_single), axis=0)

# Reconstruct levels
curr_single = kt_common[train_split_idx - 1]
pred_levels_single = []
for t in range(len(y_pred_single)):
    pred_diff = scaler_single.inverse_transform(y_pred_single[t:t+1])[0, 0]
    curr_single = curr_single + pred_diff + bias_single[0]
    pred_levels_single.append(curr_single)
pred_levels_single = np.array(pred_levels_single)

true_kt_val = kt_common[train_split_idx:train_split_idx + len(y_pred_single)]
rmse_single = np.sqrt(mean_squared_error(true_kt_val, pred_levels_single))

print(f'\n--- ABLATION 3: Multi-Population (7D) vs Single-Population (1D) ---')
print(f'Multi-Population (7 factors) RMSE on Kt: {rmse_diffs_champion:.4f}')
print(f'Single-Population (Kt only) RMSE on Kt: {rmse_single:.4f}')
improvement_multi = (rmse_single - rmse_diffs_champion) / rmse_single * 100
print(f'Improvement from multi-population training: +{improvement_multi:.1f}%')

[Reproducibility] Global seed set to 42
Training single-population model (Kt only)...

--- ABLATION 3: Multi-Population (7D) vs Single-Population (1D) ---
Multi-Population (7 factors) RMSE on Kt: 15.8981
Single-Population (Kt only) RMSE on Kt: 4.2591
Improvement from multi-population training: +-273.3%


## 6.5: Summary Table

In [5]:
print('\n' + '='*70)
print('ABLATION STUDY SUMMARY')
print('='*70)
print(f'{"Configuration":<45} {"RMSE (Kt)":<12} {"vs Champion"}')
print('-'*70)
print(f'{"Champion (Diffs + MBC + Multi-Pop)":<45} {rmse_diffs_champion:<12.4f} {"(baseline)"}')
print(f'{"Ablation 1: Levels (no differencing)":<45} {rmse_levels:<12.4f} {improvement:+.1f}% worse')
print(f'{"Ablation 2: No MBC":<45} {rmse_no_mbc:<12.4f} {improvement_mbc:+.1f}% worse')
print(f'{"Ablation 3: Single-Population (Kt only)":<45} {rmse_single:<12.4f} {improvement_multi:+.1f}% worse')
print('='*70)



ABLATION STUDY SUMMARY
Configuration                                 RMSE (Kt)    vs Champion
----------------------------------------------------------------------
Champion (Diffs + MBC + Multi-Pop)            15.8981      (baseline)
Ablation 1: Levels (no differencing)          30.9305      +48.6% worse
Ablation 2: No MBC                            19.5271      +18.6% worse
Ablation 3: Single-Population (Kt only)       4.2591       -273.3% worse
